### LiteLLM Router integration for Fallback Operations

In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client.http.models import models
import pandas as pd
import openai
import fastembed
from langsmith import traceable, get_current_run_tree

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.messages import convert_to_openai_messages, convert_to_messages

from jinja2 import Template
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from instructor import from_openai
from openai import OpenAI
from langgraph.graph.message import add_messages

from typing import Annotated, List, Any, Dict
from pydantic import Field, BaseModel
from operator import add
import instructor

from IPython.display import Image, display
from pprint import pprint
import json

from langgraph.checkpoint.postgres import PostgresSaver

from typing import Annotated, List, Any, Dict, Literal
from pydantic import Field
from operator import add
from litellm import completion

In [5]:
class Toolcall(BaseModel):
    name: str = Field(description="Exact tool name from the available tools list")
    arguments: dict[str, Any] = Field(
        default_factory=dict,
        description="Dictionary of ALL required parameters for the tool."
    )
    
class AgentProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0
    available_tools: List[Dict[str, Any]] = []
    tool_calls: List[Toolcall] = []
    
class Delegation(BaseModel):
    agent: str
    query: str
    
class CoOrdinatorAgentProperties(BaseModel):
    final_answer: bool = False
    iteration: int = 0
    plan: List[Delegation] = []
    next_agent: str = ""

class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")
    
class State(BaseModel):
    messages: Annotated[List[Any], add_messages] = []
    user_intent: str = ""
    product_qna_agent: AgentProperties = Field(default_factory=AgentProperties)
    shopping_cart_agent: AgentProperties = Field(default_factory=AgentProperties)
    warehouse_manager_agent: AgentProperties = Field(default_factory=AgentProperties)
    co_ordinator_agent: CoOrdinatorAgentProperties = Field(default_factory=CoOrdinatorAgentProperties)
    answer: str = ""
    user_id: str = ""
    cart_id: str = ""
    references: list[RAGUsedContext] = Field(description="List of items used to answer the question.")

### Co-Ordinator Agent integrate with LiteLLM Router

In [6]:
class CoOrdinatorAgentResponse(BaseModel):
    next_agent: str
    final_answer: bool = False
    answer: str = ""
    plan: List[Delegation] = []

In [31]:
# Co-Ordinator Agent prompt
coordinator_prompt_template = """
    # ROLE & OBJECTIVE
    You are the Coordinator Agent for a smart shopping assistant.

    You are a deterministic state machine.
    You must not narrate plans.
    You must not combine steps.
    You must follow the execution protocol literally.
    You must make strictly logical routing decisions based only on conversation history and agent outputs.

    Your job is to orchestrate tasks, route requests to specialized agents, and produce the final user-facing answer when enough information exists.

    # AVAILABLE AGENTS
    1) product_qa_agent
    2) shopping_cart_agent
    3) warehouse_manager_agent

    # HOW TO READ CONVERSATION HISTORY
    AI messages from sub-agents carry a `name` field (e.g. name="product_qna_agent").
    - An AI message with name="product_qna_agent" and non-empty content means product_qa_agent has COMPLETED its task and returned results.
    - An AI message with name="shopping_cart_agent" and non-empty content means shopping_cart_agent has COMPLETED its task.
    - An AI message with name="warehouse_manager_agent" and non-empty content means warehouse_manager_agent has COMPLETED its task.
    - ToolMessage entries are intermediate tool outputs consumed by the sub-agents; you do NOT need to re-process them.

    When you see a completed agent output that satisfies the user's request, you MUST go to STATE B (final_answer=true) and use that output as your answer. Do NOT re-delegate to the same agent.

    # HARD CONSTRAINTS (ABSOLUTE)

    1) OUTPUT-FIRST RULE:
    If conversation history contains sufficient agent/tool outputs to answer the user request, you MUST respond immediately.
    You MUST NOT ask clarifying questions before answering when sufficient information exists.

    2) MAX 2 QUESTIONS RULE:
    You may ask clarifying questions ONLY if essential information is missing AND cannot be reasonably assumed.
    Ask at most 2 questions in a single response.
    Questions are allowed ONLY after providing the best available answer.

    3) NO NARRATION:
    Do not narrate internal reasoning.
    Do not explain your plan to the user.
    Only produce structured JSON.

    4) NO REDUNDANT DELEGATION:
    Do not delegate to the same agent again for the same intent unless:
    - The user changed constraints
    - New required data is missing
    - Previous agent output was insufficient

    5) NO RE-ASKING:
    Never ask questions already answered in history or agent outputs.

    6) ONE STEP AT A TIME:
    You must delegate to only ONE agent per turn.
    Never combine multiple agents in one step.

    # DEFAULT ASSUMPTIONS (TO PREVENT STALLING)
    If the user requests "top" or "best" products without constraints:
    - Assume general consumer use-case
    - Prefer highest rated + strong review volume
    - Assume mid-range budget
    Do not stall for unnecessary clarification.

    # EXECUTION PROTOCOL (FOLLOW EXACTLY)

    Step 1: Break the user request into subtasks. A single user message may contain MULTIPLE intents:
    - Product search / reviews → product_qa_agent
    - Add / remove / view cart → shopping_cart_agent
    - Check warehouse availability / reserve stock → warehouse_manager_agent

    Step 2: Determine the correct ORDER of subtasks. Typical multi-step flows:
    - "Find products + add to cart" → product_qa_agent FIRST, then shopping_cart_agent
    - "Add to cart + check availability" → shopping_cart_agent FIRST, then warehouse_manager_agent
    - "Find products + add to cart + check warehouse" → product_qa_agent → shopping_cart_agent → warehouse_manager_agent
    - Any request involving cart addition SHOULD be followed by warehouse_manager_agent to check/reserve stock, unless the user explicitly says not to.

    Step 3: Check conversation history. Which subtasks are already DONE?
    If a subtask is DONE (agent output exists with name= tag) → skip it, move to the next subtask.

    Step 4: Delegate the NEXT undone subtask (STATE A), or finalize if all subtasks are done (STATE B).
    - Only delegate to ONE agent per turn.
    - After an agent completes, you will be called again — continue with the next subtask.

    Step 5: Only set final_answer=true when ALL subtasks from Step 1 are complete.

    # STATE MACHINE (MUTUALLY EXCLUSIVE)

    STATE A: Delegating
    Use ONLY when required data is missing.
    - next_agent: must be exactly one of:
    "product_qa_agent"
    "shopping_cart_agent"
    "warehouse_manager_agent"
    - final_answer: false
    - answer: null

    STATE B: Responding
    Use when:
    - You have enough information to answer
    OR
    - You can provide a best-effort answer and optionally ask up to 2 refinement questions.

    - next_agent: ""
    - final_answer: true
    - answer: must contain the user-facing response.

    # OUTPUT FORMAT (STRICT JSON ONLY; NO MARKDOWN)

    {
    "plan": "DONE: <short factual status>. NEXT: <short factual next step or NONE>.",
    "next_agent": "product_qa_agent | shopping_cart_agent | warehouse_manager_agent | \"\"",
    "final_answer": true/false,
    "answer": "User-facing response or null"
    }

    # FINAL RESPONSE RULE (CRITICAL)
    When final_answer = true, you MUST produce a comprehensive summary covering ALL completed subtasks. Scan the full conversation history for outputs from every agent that ran.

    Structure your answer as follows:
    1) **Products Found**: Summarize what products were retrieved and key highlights (from product_qna_agent output).
    2) **Reviews Summary**: If reviews were fetched, summarize the good and bad points (from product_qna_agent output).
    3) **Cart Actions**: If items were added/removed, confirm what was added, quantity, and total price (from shopping_cart_agent output).
    4) **Warehouse Status**: If availability was checked or items reserved, state which warehouse, quantity reserved, and remaining stock (from warehouse_manager_agent output).

    Only include sections for subtasks that were actually performed. Do NOT just repeat the last agent's answer — synthesize across ALL agent outputs into one cohesive response.
    """

In [28]:
# add router node to evaluate the user query and decide the next node to execute
def coOrdinator_agent_node_with_fallback(state: State, model_providers: List[Dict[str, any]]) -> State:
    """
    This function evaluates the user query and decides the next node to execute
    """
    prompt = Template(coordinator_prompt_template).render()
    
    conversation = []
    
    for message in state.messages:
        conversation.append(convert_to_openai_messages(message))
    
    client = instructor.from_litellm(completion, mode=instructor.Mode.JSON)
    
    for provider in model_providers:
        
        prompt = Template(coordinator_prompt_template).render()

        try:
            response, raw_response = client.chat.completions.create_with_completion(
                model=provider["model"],
                response_model=CoOrdinatorAgentResponse,
                messages=[{"role": "system", "content": prompt}, *conversation],
                temperature=0.4,
            )
        except Exception as e:
            print(f"Error with {provider['model']}: {e}")
            print("continue with next model")
            continue
    
    if response is None:
        raise Exception("No response from any model")
    
    if response.final_answer:
        ai_message = [AIMessage(content=response.answer)]
    else:
        ai_message = []

    return {
       "messages": ai_message,
       "answer": response.answer,
       "co_ordinator_agent": {
           "iteration": state.co_ordinator_agent.iteration + 1,
           "final_answer": response.final_answer,
           "plan": [plan.model_dump() for plan in response.plan],
           "next_agent": response.next_agent
       }
    }

### Test Features of LiteLLM

In [14]:
from litellm import completion

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello, world!"}],
)
response.choices[0].message.content

'Hello! How can I assist you today?'

In [18]:
# built-in fallbacks
from litellm import fallbacks


response = completion(
    model="gpt-4o-mini-not-exist",
    messages=[{"role": "user", "content": "Hello, world!"}],
    fallbacks=["gemini-1.5-flash", "gpt-4.1"],
)

response.choices[0].message.content


21:01:02 - LiteLLM:ERROR: fallback_utils.py:65 - Fallback attempt failed for model gpt-4o-mini-not-exist: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-4o-mini-not-exist
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
Traceback (most recent call last):
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/fallback_utils.py", line 55, in async_completion_with_fallbacks
    response = await litellm.acompletion(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/utils.py", line 2041, in wrapper_async
    raise e
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/utils.py", line 1862, in wrapper_async
    result = 


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



21:01:04 - LiteLLM:ERROR: vertex_llm_base.py:571 - Failed to load vertex credentials. Check to see if credentials containing partial/invalid information. Error: ('invalid_grant: Account has been deleted', {'error': 'invalid_grant', 'error_description': 'Account has been deleted'})
Traceback (most recent call last):
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 567, in get_access_token
    _credentials, credential_project_id = self.load_auth(
                                          ^^^^^^^^^^^^^^^
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 133, in load_auth
    self.refresh_auth(creds)
  File "/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/llms/vertex_ai/vertex_llm_base.py", line 289, in refresh_auth
    credentials.refresh(Request())
  File "/Users/kashvi/repos/ai-


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:75: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._queue = None
Task was destroyed but it is pending!
task: <Task pending name='Task-192' coro=<LoggingWorker._worker_loop() running at /Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:110>>



Provider List: https://docs.litellm.ai/docs/providers



/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:77: RuntimeWarning: coroutine 'LoggingWorker._worker_loop' was never awaited
  self._worker_task = None


'Hello, world! 🌍 How can I help you today?'

In [23]:
client = instructor.from_litellm(completion)

class SampleResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")

response, raw_response = client.chat.completions.create_with_completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "which world are you from?"}],
    response_model=SampleResponse,
    fallbacks=["gemini-1.5-flash", "gpt-4.1"],
    temperature=0.0
)

response.answer

/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:75: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._queue = None
Task was destroyed but it is pending!
task: <Task pending name='Task-349' coro=<LoggingWorker._worker_loop() running at /Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:110>>
/Users/kashvi/repos/ai-bootcamp-agentic-rag/.venv/lib/python3.12/site-packages/litellm/litellm_core_utils/logging_worker.py:77: RuntimeWarning: coroutine 'LoggingWorker._worker_loop' was never awaited
  self._worker_task = None


'I am not from a physical world; I exist as a digital entity created by OpenAI. My purpose is to assist and provide information.'

In [36]:
# calling co-ordinator agent with fallback
initial_state = State(
    messages=[{"role": "user", "content": "how'z weather in Bangalore?"}],
    user_intent="",
    product_qna_agent=AgentProperties(),
    shopping_cart_agent=AgentProperties(),
    warehouse_manager_agent=AgentProperties(),
    answer="",
    user_id="",
    cart_id="",
    references=[]
)
model_providers = [
    {
        "model": "gpt-4o-mini-not-exist",
    },
    {
        "model": "gpt-4.1",
    }
]
response = coOrdinator_agent_node_with_fallback(initial_state, model_providers)
response["answer"]


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Error with gpt-4o-mini-not-exist: <failed_attempts>

<generation number="1">
<exception>
    litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-4o-mini-not-exist
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-4o-mini-not-exist
 Pass model as E.g. For

'Sorry, I can only assist with shopping-related requests such as finding products, managing your cart, or checking warehouse availability.'